In [381]:
import numpy as np
import pandas as pd
import torch
from torch import nn 

In [382]:
def train_test_split(X, y, test_size:float = None, train_size:float = None):
	if X.shape[0] != y.shape[0]:
		raise Exception("Size mismatch of X and y")
	else:
		total_rows=X.shape[0]
	if test_size == None and train_size == None:
		test_size=0.25
		train_size = 1 - test_size
	elif test_size == None:
		test_size = 1 - train_size
	elif train_size == None:
		train_size = 1 - test_size

	mark = int(total_rows*train_size)
	return X[:mark], y[:mark], X[mark:], y[mark:]


In [383]:
# from sklearn.model_selection import train_test_split
df = pd.read_csv('data.csv')
df.head()
X = df.drop(columns=['price']).to_numpy()
y = df['price'].to_numpy()

In [384]:
X_train, y_train, X_test, y_test = train_test_split(X, y, 0.2)
# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train

array([[-1.4598444 , -1.78017249],
       [-1.20410524, -0.83074716],
       [-0.69262691, -0.83074716],
       [-0.18114858,  0.11867817],
       [ 0.33032976,  0.11867817],
       [ 0.67131531,  1.06810349],
       [ 1.01230086,  1.06810349],
       [ 1.52377919,  1.06810349]])

In [391]:
class Perceptron():
	def __init__(self, features_size):
		self.weights = torch.tensor(np.random.random(features_size), requires_grad=True, dtype=torch.float32)
		self.bias = torch.tensor(np.random.random(), requires_grad=True, dtype=torch.float32)

	def forward(self, features):
		features = torch.tensor(features, dtype=torch.float32)
		# self.model_pred = torch.empty(features.shape[0], dtype=torch.float32)
		self.model_pred = (features @ self.weights) + self.bias
		return self.model_pred
			
	def rmse(self, y_true):
		y_true = torch.tensor(y_true, dtype=torch.float32)
		if y_true.shape != self.model_pred.shape:
			raise Exception("Shape mismatch(y_true and y_pred are not same shape)")
		num = y_true.numel()
		self.loss = torch.mean((y_true-self.model_pred)**2).sqrt()
		return self.loss
	
	def backpropagation(self, lr=0.01):
		with torch.no_grad():
			# for i in range(len(self.weights)):
			# 	self.weights[i] = self.weights[i] - (lr * self.weights.grad[i])
			self.weights -= lr * self.weights.grad
			self.bias -= (lr * self.bias.grad)




In [398]:
model = Perceptron(X_train.shape[1])
epochs = 100
for i in range(epochs):
	y_pred = (model.forward(X_train))

	# Calculate loss

	loss = model.rmse(y_train)
	print("loss:", loss)

	model.weights.grad = None
	model.bias.grad=None
	# Calculating loss
	model.loss.backward()

	# update weights 
	model.backpropagation(lr=100)

	print("After backprop weights:", model.weights)
	print("After backprop bias:", model.bias)
	

# print(model.bias.grad)
# print(model.bias)



loss: tensor(147169.8438, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([43.2772, 41.4245], requires_grad=True)
After backprop bias: tensor(91.2971, requires_grad=True)
loss: tensor(147053.4219, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([85.8422, 81.9726], requires_grad=True)
After backprop bias: tensor(181.7633, requires_grad=True)
loss: tensor(146937.0312, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([128.3857, 122.4976], requires_grad=True)
After backprop bias: tensor(272.2397, requires_grad=True)
loss: tensor(146820.6562, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([170.9077, 162.9995], requires_grad=True)
After backprop bias: tensor(362.7261, requires_grad=True)
loss: tensor(146704.2969, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([213.4081, 203.4784], requires_grad=True)
After backprop bias: tensor(453.2226, requires_grad=True)
loss: tensor(146587.9531, grad_fn=<SqrtBackward0>)
After backprop weights: tensor([255.8870, 24

In [387]:
with torch.no_grad():
	model.weights[0] -= 0.1*model.weights.grad[0]
	print(model.weights)



tensor([4.6396, 4.5645], requires_grad=True)


In [388]:
with torch.no_grad():
	model.weights[0] = 2 - 0.01 * -4.7684e-07

model.weights[0]

tensor(2., grad_fn=<SelectBackward0>)

In [389]:
y_t = np.array([1])
y_p = np.array([0.5])
((sum((y_t - y_p)**2))**0.5)/len(y_t)


np.float64(0.5)

In [390]:
t = torch.empty(X_train.shape[0])
for i in range(5):
	t[i] += i
t

tensor([-14.7021,  -8.2422,  -4.9128,   2.7119,   6.0413,   7.8895,   9.4425,
         11.7720])